In [8]:
import re
import requests
import pandas as pd
from bs4 import BeautifulSoup
from urllib.parse import urljoin
from datetime import datetime

In [9]:
start = "2016-01-01"
end = "2026-12-31"

series = {
    "fed_target_upper": "DFEDTARU",
    "fed_target_lower": "DFEDTARL",
    "effective_ffr": "DFF",
}

dfs = []

for name, code in series.items():
    url = f"https://fred.stlouisfed.org/graph/fredgraph.csv?id={code}"
    df = pd.read_csv(url)

    # 关键：FRED 的日期列叫 observation_date
    df = df.rename(columns={"observation_date": "DATE"})

    df["DATE"] = pd.to_datetime(df["DATE"])
    df[code] = pd.to_numeric(df[code], errors="coerce")

    df = df.rename(columns={code: name})
    df = df[(df["DATE"] >= start) & (df["DATE"] <= end)]

    dfs.append(df[["DATE", name]])

rate_df = dfs[0]
for df in dfs[1:]:
    rate_df = rate_df.merge(df, on="DATE", how="outer")

rate_df = rate_df.sort_values("DATE").reset_index(drop=True)

rate_df.head()

,DATE,fed_target_upper,fed_target_lower,effective_ffr
0,2016-01-01,0.5,0.25,0.20
1,2016-01-02,0.5,0.25,0.20
2,2016-01-03,0.5,0.25,0.20
3,2016-01-04,0.5,0.25,0.36
4,2016-01-05,0.5,0.25,0.36


In [10]:
START_DATE = "2016-01-01"
END_DATE = "2026-12-31"

BASE = "https://www.federalreserve.gov"

HEADERS = {
    "User-Agent": "Mozilla/5.0"
}


In [11]:
def load_fred_series(series_dict, start_date, end_date):
    """
    从 FRED 下载多个 series，并合并成一个 DataFrame。

    series_dict 例如：
    {
        "fed_target_upper": "DFEDTARU",
        "fed_target_lower": "DFEDTARL",
        "effective_ffr": "DFF",
    }
    """

    dfs = []

    for name, code in series_dict.items():
        url = f"https://fred.stlouisfed.org/graph/fredgraph.csv?id={code}"
        df = pd.read_csv(url)

        # FRED 日期列通常叫 observation_date
        if "observation_date" in df.columns:
            df = df.rename(columns={"observation_date": "DATE"})
        elif "DATE" not in df.columns:
            raise ValueError(f"{code} 找不到日期列，当前列名是: {df.columns.tolist()}")

        df["DATE"] = pd.to_datetime(df["DATE"])
        df[code] = pd.to_numeric(df[code], errors="coerce")

        df = df.rename(columns={code: name})
        df = df[(df["DATE"] >= start_date) & (df["DATE"] <= end_date)]

        dfs.append(df[["DATE", name]])

    rate_df = dfs[0]

    for df in dfs[1:]:
        rate_df = rate_df.merge(df, on="DATE", how="outer")

    rate_df = rate_df.sort_values("DATE").reset_index(drop=True)

    return rate_df


series = {
    "fed_target_upper": "DFEDTARU",
    "fed_target_lower": "DFEDTARL",
    "effective_ffr": "DFF",
}

rate_df = load_fred_series(series, START_DATE, END_DATE)

print("Daily rate data:")
rate_df.head()

Daily rate data:


,DATE,fed_target_upper,fed_target_lower,effective_ffr
0,2016-01-01,0.5,0.25,0.20
1,2016-01-02,0.5,0.25,0.20
2,2016-01-03,0.5,0.25,0.20
3,2016-01-04,0.5,0.25,0.36
4,2016-01-05,0.5,0.25,0.36


In [12]:
def get_policy_changes(rate_df):
    """
    根据 fed_target_upper / fed_target_lower 的变化，
    找出政策目标区间实际变化日。
    """

    df = rate_df.copy()

    df["upper_change"] = df["fed_target_upper"].diff()
    df["lower_change"] = df["fed_target_lower"].diff()

    policy_changes = df[
        (df["upper_change"].fillna(0) != 0) |
        (df["lower_change"].fillna(0) != 0)
    ].copy()

    policy_changes = policy_changes[
        [
            "DATE",
            "fed_target_lower",
            "fed_target_upper",
            "effective_ffr",
            "lower_change",
            "upper_change",
        ]
    ].reset_index(drop=True)

    policy_changes = policy_changes.rename(columns={"DATE": "effective_date"})

    return policy_changes


policy_changes = get_policy_changes(rate_df)

print("\nPolicy changes from FRED:")
policy_changes.head()



Policy changes from FRED:


,effective_date,fed_target_lower,fed_target_upper,effective_ffr,lower_change,upper_change
0,2016-12-14,0.50,0.75,0.41,0.25,0.25
1,2017-03-16,0.75,1.00,0.91,0.25,0.25
2,2017-06-15,1.00,1.25,1.16,0.25,0.25
3,2017-12-14,1.25,1.50,1.41,0.25,0.25
4,2018-03-22,1.50,1.75,1.68,0.25,0.25


In [13]:
import re
from urllib.parse import urljoin

import pandas as pd
import requests
from bs4 import BeautifulSoup


def get_soup(url):
    r = requests.get(url, headers=HEADERS, timeout=30)
    r.raise_for_status()
    return BeautifulSoup(r.text, "html.parser")


def url_exists(url):
    """
    ?????? HEAD ?????????? GET????????
    """
    if not url:
        return False
    try:
        r = requests.get(url, headers=HEADERS, timeout=20, stream=True)
        return r.status_code == 200
    except Exception:
        return False


def scrape_fomc_press_release_year(year):
    """
    ???? FOMC press release ???
    https://www.federalreserve.gov/newsevents/pressreleases/YYYY-press-fomc.htm, with fallback to YYYY-press.htm

    ?????? FOMC statement???? economic projections ??????
    """
    page_candidates = [
        f"{BASE}/newsevents/pressreleases/{year}-press-fomc.htm",
        f"{BASE}/newsevents/pressreleases/{year}-press.htm",
    ]

    soup = None
    page_url = None
    errors = []
    for candidate_url in page_candidates:
        try:
            soup = get_soup(candidate_url)
            page_url = candidate_url
            break
        except Exception as e:
            errors.append(f"{candidate_url}: {e}")

    if soup is None:
        print(f"Warning: cannot open Federal Reserve press release pages for {year}: {'; '.join(errors)}")
        return []

    records = []

    # Fed ???? press release ????? row??? time??? event?
    for row in soup.select("div.row"):
        time_tag = row.select_one("time")
        if time_tag is None:
            continue

        date_text = time_tag.get_text(" ", strip=True)
        if not re.fullmatch(r"\d{1,2}/\d{1,2}/\d{4}", date_text):
            continue

        event_div = row.select_one(".eventlist__event") or row
        links = event_div.find_all("a", href=True)

        for link in links:
            title = " ".join(link.get_text(" ", strip=True).split())
            if title != "Federal Reserve issues FOMC statement":
                continue

            announcement_date = pd.to_datetime(date_text, format="%m/%d/%Y")
            statement_url = urljoin(BASE, link["href"])

            # implementation note ?????? statement URL ??? 1??? monetary20250319a1.htm?
            implementation_note_url = statement_url.replace("a.htm", "a1.htm")
            if not url_exists(implementation_note_url):
                implementation_note_url = None

            records.append(
                {
                    "year": year,
                    "announcement_date": announcement_date,
                    "statement_title": title,
                    "statement_url": statement_url,
                    "implementation_note_url": implementation_note_url,
                    "source_page": page_url,
                }
            )

    # ???????????????????? + ???????
    if not records:
        text = soup.get_text("\n", strip=True)
        lines = [x.strip() for x in text.split("\n") if x.strip()]
        for i, line in enumerate(lines):
            if not re.fullmatch(r"\d{1,2}/\d{1,2}/\d{4}", line):
                continue
            nearby = " ".join(lines[i:i+5])
            if "Federal Reserve issues FOMC statement" not in nearby:
                continue
            announcement_date = pd.to_datetime(line, format="%m/%d/%Y")
            ymd = announcement_date.strftime("%Y%m%d")
            statement_url = f"{BASE}/newsevents/pressreleases/monetary{ymd}a.htm"
            implementation_note_url = f"{BASE}/newsevents/pressreleases/monetary{ymd}a1.htm"
            records.append(
                {
                    "year": year,
                    "announcement_date": announcement_date,
                    "statement_title": "Federal Reserve issues FOMC statement",
                    "statement_url": statement_url if url_exists(statement_url) else None,
                    "implementation_note_url": implementation_note_url if url_exists(implementation_note_url) else None,
                    "source_page": page_url,
                }
            )

    return records


def scrape_fomc_events(start_date, end_date):
    start_year = pd.to_datetime(start_date).year
    end_year = pd.to_datetime(end_date).year

    records = []

    for year in range(start_year, end_year + 1):
        year_records = scrape_fomc_press_release_year(year)
        print(f"{year}: {len(year_records)} FOMC statement events")
        records.extend(year_records)

    events = pd.DataFrame(records)

    if events.empty:
        raise ValueError(
            "No FOMC events were scraped. Check network access or Federal Reserve page structure."
        )

    events = events[
        (events["announcement_date"] >= pd.to_datetime(start_date)) &
        (events["announcement_date"] <= pd.to_datetime(end_date))
    ].copy()

    events = (
        events.drop_duplicates(subset=["announcement_date"])
              .sort_values("announcement_date")
              .reset_index(drop=True)
    )

    if events.empty:
        raise ValueError(
            f"FOMC events were scraped, but none fall between {start_date} and {end_date}."
        )

    return events


fomc_events = scrape_fomc_events(START_DATE, END_DATE)

print("FOMC events:")
print(fomc_events.head())

2016: 8 FOMC statement events
2017: 8 FOMC statement events
2018: 8 FOMC statement events
2019: 8 FOMC statement events
2020: 12 FOMC statement events
2021: 10 FOMC statement events
2022: 10 FOMC statement events
2023: 10 FOMC statement events
2024: 10 FOMC statement events
2025: 10 FOMC statement events
2026: 5 FOMC statement events
FOMC events:
   year announcement_date                        statement_title  \
0  2016        2016-01-27  Federal Reserve issues FOMC statement   
1  2016        2016-03-16  Federal Reserve issues FOMC statement   
2  2016        2016-04-27  Federal Reserve issues FOMC statement   
3  2016        2016-06-15  Federal Reserve issues FOMC statement   
4  2016        2016-07-27  Federal Reserve issues FOMC statement   

                                       statement_url  \
0  https://www.federalreserve.gov/newsevents/pres...   
1  https://www.federalreserve.gov/newsevents/pres...   
2  https://www.federalreserve.gov/newsevents/pres...   
3  https://www.fed

In [14]:
def parse_effective_date_from_implementation_note(url):
    """
    从 implementation note 页面里解析 Effective Date。

    常见文本：
    Effective March 17, 2022
    Effective July 27, 2023
    """

    if pd.isna(url) or url is None:
        return pd.NaT

    try:
        soup = get_soup(url)
        text = soup.get_text(" ", strip=True)

        m = re.search(
            r"Effective\s+([A-Za-z]+\s+\d{1,2},\s+\d{4})",
            text
        )

        if m:
            return pd.to_datetime(m.group(1))

    except Exception as e:
        print(f"Warning: failed to parse effective date from {url}, error = {e}")

    return pd.NaT


fomc_events["note_effective_date"] = fomc_events["implementation_note_url"].apply(
    parse_effective_date_from_implementation_note
)

fomc_events["note_days_to_effective"] = (
    fomc_events["note_effective_date"] - fomc_events["announcement_date"]
).dt.days

print("\nFOMC events with parsed effective date:")
print(fomc_events.head())




FOMC events with parsed effective date:
   year announcement_date                        statement_title  \
0  2016        2016-01-27  Federal Reserve issues FOMC statement   
1  2016        2016-03-16  Federal Reserve issues FOMC statement   
2  2016        2016-04-27  Federal Reserve issues FOMC statement   
3  2016        2016-06-15  Federal Reserve issues FOMC statement   
4  2016        2016-07-27  Federal Reserve issues FOMC statement   

                                       statement_url  \
0  https://www.federalreserve.gov/newsevents/pres...   
1  https://www.federalreserve.gov/newsevents/pres...   
2  https://www.federalreserve.gov/newsevents/pres...   
3  https://www.federalreserve.gov/newsevents/pres...   
4  https://www.federalreserve.gov/newsevents/pres...   

                             implementation_note_url  \
0  https://www.federalreserve.gov/newsevents/pres...   
1  https://www.federalreserve.gov/newsevents/pres...   
2  https://www.federalreserve.gov/newsevents/

In [15]:
def build_daily_policy_rate_df(rate_df: pd.DataFrame, fomc_events: pd.DataFrame) -> pd.DataFrame:
    """
    Build the final daily policy-rate panel.

    Columns:
    - date
    - effective_ffr: actual effective federal funds rate
    - fed_target_lower / fed_target_upper: target range
    - is_announcement_date: True on FOMC statement announcement dates
    - is_effective_date: True on FOMC implementation-note effective dates; target-change dates are included as fallback
    - target_change_direction: up / down / unchanged
    - target_change_bps: daily change in target midpoint, in basis points
    """
    df = rate_df.copy()
    df["DATE"] = pd.to_datetime(df["DATE"])
    df = df.sort_values("DATE").reset_index(drop=True)

    # Target midpoint is used to classify whether the policy range moved up or down.
    df["target_midpoint"] = (df["fed_target_lower"] + df["fed_target_upper"]) / 2.0
    df["target_change_bps"] = df["target_midpoint"].diff().fillna(0.0) * 100.0

    df["target_change_direction"] = "unchanged"
    df.loc[df["target_change_bps"].gt(0), "target_change_direction"] = "up"
    df.loc[df["target_change_bps"].lt(0), "target_change_direction"] = "down"

    announcement_dates = set(pd.to_datetime(fomc_events["announcement_date"]).dt.normalize())
    df["is_announcement_date"] = df["DATE"].dt.normalize().isin(announcement_dates)

    note_effective_dates = set(
        pd.to_datetime(fomc_events["note_effective_date"], errors="coerce")
        .dropna()
        .dt.normalize()
    ) if "note_effective_date" in fomc_events.columns else set()
    target_change_dates = set(df.loc[df["target_change_bps"].ne(0), "DATE"].dt.normalize())
    effective_dates = note_effective_dates | target_change_dates
    df["is_effective_date"] = df["DATE"].dt.normalize().isin(effective_dates)

    final_df = df.rename(columns={"DATE": "date"})[
        [
            "date",
            "effective_ffr",
            "fed_target_lower",
            "fed_target_upper",
            "is_announcement_date",
            "is_effective_date",
            "target_change_direction",
            "target_change_bps",
        ]
    ].copy()

    final_df["target_change_bps"] = final_df["target_change_bps"].round(6)
    return final_df


final_df = build_daily_policy_rate_df(rate_df, fomc_events)

parquet_path = "fomc_policy_rate_daily_2016_2026.parquet"
final_df.to_parquet(parquet_path, index=False)

print(f"Saved final daily policy-rate df to {parquet_path}")
print("\nRows / columns:", final_df.shape)
print("\nFOMC effective dates and target-change dates:")
print(final_df[final_df["is_effective_date"]].to_string(index=False))

final_df.head()

Saved final daily policy-rate df to fomc_policy_rate_daily_2016_2026.parquet

Rows / columns: (3775, 8)

FOMC effective dates and target-change dates:
      date  effective_ffr  fed_target_lower  fed_target_upper  is_announcement_date  is_effective_date target_change_direction  target_change_bps
2016-01-28           0.38              0.25              0.50                 False               True               unchanged                0.0
2016-03-17           0.37              0.25              0.50                 False               True               unchanged                0.0
2016-04-28           0.37              0.25              0.50                 False               True               unchanged                0.0
2016-06-16           0.38              0.25              0.50                 False               True               unchanged                0.0
2016-07-28           0.40              0.25              0.50                 False               True               un

,date,effective_ffr,fed_target_lower,fed_target_upper,is_announcement_date,is_effective_date,target_change_direction,target_change_bps
0,2016-01-01,0.20,0.25,0.5,False,False,unchanged,0.0
1,2016-01-02,0.20,0.25,0.5,False,False,unchanged,0.0
2,2016-01-03,0.20,0.25,0.5,False,False,unchanged,0.0
3,2016-01-04,0.36,0.25,0.5,False,False,unchanged,0.0
4,2016-01-05,0.36,0.25,0.5,False,False,unchanged,0.0
